# NetCDF to PostgreSQL Ingestion

This notebook loads data from a NetCDF file using `NetCDFViewer` and bulk-inserts it into the `climate_grid` table in PostgreSQL.

The ingestion is idempotent: running it twice on the same file will not create duplicates (enforced by the `UNIQUE (time, lat, lon, variable)` constraint).

# Setup

In [ ]:
from pathlib import Path
import psycopg2
import psycopg2.extras
import numpy as np
import sys
import xarray as xr
import pandas as pd

# Database connection parameters
DB_PARAMS = {
    'host': 'localhost',
    'user': 'leon',
    'password': 'leon',
    'database': 'oc-database',
    'port': 5432
}

# Path to the NetCDF file
NC_FILE = Path("../local_only/1989.monthly_rain.nc")

# NetCDFViewer Class

In [ ]:
class NetCDFViewer:
    """Context manager for exploring and extracting data from NetCDF files."""
    
    def __init__(self, path):
        """Initialize with a path to a NetCDF file (doesn't open until context manager entry)."""
        self.path = Path(path)
        self.ds = None
    
    def __enter__(self):
        """Open the dataset on context manager entry."""
        self.ds = xr.open_dataset(self.path, engine="netcdf4")
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Close the dataset on context manager exit."""
        if self.ds is not None:
            self.ds.close()
        return False
    
    def get_spatial_slice(self, time, variable=None):
        """Extract a spatial slice for a given time step.
        
        Args:
            time: ISO string or pandas.Timestamp
            variable: Variable name. If None, auto-selects if only one real data variable exists.
        
        Returns:
            xarray.DataArray: 2D spatial grid for the given time.
        """
        if self.ds is None:
            raise RuntimeError("Dataset not open. Use 'with NetCDFViewer(path) as viewer:'")
        
        if variable is None:
            numeric_vars = [
                name for name, var in self.ds.data_vars.items()
                if var.dtype not in [object, str] and len(var.dims) > 0
            ]
            
            if len(numeric_vars) == 0:
                raise ValueError("No numeric data variables found in the dataset.")
            elif len(numeric_vars) == 1:
                variable = numeric_vars[0]
            else:
                raise ValueError(
                    f"Multiple data variables found: {numeric_vars}. "
                    f"Please specify which one to extract."
                )
        
        sliced = self.ds[variable].sel(time=time, method="nearest")
        return sliced
    
    def get_all_times(self):
        """Return all time coordinates from the dataset."""
        if self.ds is None:
            raise RuntimeError("Dataset not open. Use 'with NetCDFViewer(path) as viewer:'")
        return self.ds.coords['time'].values

# Ingestion Function

In [ ]:
def ingest_netcdf_file(nc_file_path, db_params, variable_name=None, batch_size=10000):
    """
    Ingest a NetCDF file into the climate_grid table.
    
    Args:
        nc_file_path: Path to the NetCDF file
        db_params: Database connection parameters dict
        variable_name: Specific variable to ingest (if None, auto-detects)
        batch_size: Number of rows to insert per batch (tune for performance)
    """
    
    conn = psycopg2.connect(**db_params)
    cur = conn.cursor()
    
    try:
        with NetCDFViewer(nc_file_path) as viewer:
            # Get all time steps and the variable name
            times = viewer.get_all_times()
            print(f"Found {len(times)} time steps")
            
            # If variable_name not specified, extract it from the first spatial slice
            if variable_name is None:
                with xr.open_dataset(nc_file_path, engine="netcdf4") as ds:
                    numeric_vars = [
                        name for name, var in ds.data_vars.items()
                        if var.dtype not in [object, str] and len(var.dims) > 0
                    ]
                    if len(numeric_vars) == 1:
                        variable_name = numeric_vars[0]
                    else:
                        raise ValueError(f"Could not auto-detect variable. Found: {numeric_vars}")
            
            print(f"Ingesting variable: {variable_name}")
            
            # Iterate over each time step
            for time_idx, time_val in enumerate(times):
                # Extract spatial grid for this time step
                spatial_slice = viewer.get_spatial_slice(
                    pd.Timestamp(time_val),
                    variable=variable_name
                )
                
                # Get lat/lon coordinates
                lats = spatial_slice.coords['lat'].values
                lons = spatial_slice.coords['lon'].values
                
                # Convert the 2D spatial grid to a flat list of rows
                # Each row is (time, lat, lon, variable, value)
                rows = []
                # Convert numpy.datetime64 to Python datetime (required for psycopg2)
                time_py = pd.Timestamp(time_val).to_pydatetime()
                for i, lat in enumerate(lats):
                    for j, lon in enumerate(lons):
                        value = spatial_slice.values[i, j]
                        # Only insert non-NaN values
                        if not np.isnan(value):
                            rows.append((time_py, float(lat), float(lon), variable_name, float(value)))
                
                print(f"  Time {time_idx + 1}/{len(times)}: {time_val} -> {len(rows):,} rows")
                
                # ═══════════════════════════════════════════════════════════════════════════════
                # BULK INSERT EXPLANATION: psycopg2.extras.execute_values()
                # ═══════════════════════════════════════════════════════════════════════════════
                #
                # execute_values() is a helper from psycopg2.extras that efficiently inserts
                # large batches of rows without sending individual SQL statements to the database.
                #
                # Parameters:
                #   1. cur: the database cursor
                #   2. sql_template: SQL with %s placeholders (values go here)
                #   3. rows: list of tuples, one tuple per row
                #   4. page_size: how many rows to insert per SQL call (batching within the batch)
                #      Smaller = fewer database round-trips per batch but more SQL calls
                #      Larger = fewer calls but larger memory footprint and larger SQL strings
                #      10,000 is a good default; adjust if needed
                #
                # How it works:
                #   Instead of:
                #     for row in rows:
                #         cur.execute("INSERT INTO ... VALUES (%s, %s, ...)", row)
                #   
                #   execute_values() generates SQL like:
                #     INSERT INTO climate_grid (time, lat, lon, variable, value)
                #     VALUES (%s, %s, %s, %s, %s),
                #            (%s, %s, %s, %s, %s),
                #            ...
                #     (repeated page_size times)
                #   
                #   This drastically reduces SQL overhead and is typically 100-1000x faster
                #   than row-by-row inserts.
                #
                # ON CONFLICT DO NOTHING:
                #   If a row already exists (hits the UNIQUE constraint), PostgreSQL silently
                #   ignores it. This makes the ingestion IDEMPOTENT: you can run this notebook
                #   twice on the same file and get no duplicates.
                # ═══════════════════════════════════════════════════════════════════════════════
                
                if rows:
                    psycopg2.extras.execute_values(
                        cur,
                        """INSERT INTO climate_grid (time, lat, lon, variable, value)
                           VALUES %s
                           ON CONFLICT DO NOTHING""",
                        rows,
                        page_size=batch_size
                    )
        
        # Commit all changes
        conn.commit()
        print(f"\n✓ Ingestion complete!")
        
    except Exception as e:
        conn.rollback()
        print(f"✗ Error during ingestion: {e}")
        raise
    finally:
        cur.close()
        conn.close()

# Run Ingestion

In [ ]:
# Ingest the 1989 rainfall data
ingest_netcdf_file(NC_FILE, DB_PARAMS, batch_size=10000)

# Verify Ingestion

In [ ]:
conn = psycopg2.connect(**DB_PARAMS)
cur = conn.cursor()

# Check total row count
cur.execute("SELECT COUNT(*) FROM climate_grid")
total_rows = cur.fetchone()[0]
print(f"Total rows in climate_grid: {total_rows:,}")

# Check rows per variable
cur.execute("""
    SELECT variable, COUNT(*) as row_count
    FROM climate_grid
    GROUP BY variable
""")
print("\nRows per variable:")
for var, count in cur.fetchall():
    print(f"  {var}: {count:,}")

# Check unique time steps
cur.execute("""
    SELECT COUNT(DISTINCT time) as unique_times
    FROM climate_grid
""")
unique_times = cur.fetchone()[0]
print(f"\nUnique time steps: {unique_times}")

# Show sample data
cur.execute("""
    SELECT time, lat, lon, variable, value
    FROM climate_grid
    LIMIT 5
""")
print("\nSample rows:")
for row in cur.fetchall():
    print(f"  {row}")

cur.close()
conn.close()